# Eval Comparison: v14 (LB 0.68) vs dgxchen (LB 0.84) — Same Holdout

**Goal:** find WHERE v14 regressed. Was it uniform across categories (suggesting general template confusion from the augmenter dual-template) or concentrated in specific categories (suggesting the cipher swap broke cipher specifically, etc.)? This directly informs the v15 design.

**Method:** load base Nemotron-30B once; attach BOTH adapters via PEFT multi-adapter API; eval both on the SAME 30-problem stratified holdout (seed=42 → identical sample for both); diff per-category.

**Hypotheses going in:**
- H1 (uniform collapse): v14 underperforms dgxchen by ~20% across all categories → dual-template inference confusion is the dominant failure
- H2 (cipher-concentrated): v14 only loses cipher accuracy → our cipher v2 swap is the dominant failure
- H3 (augmenter-format leakage): v14 produces multi-line table output on reasoner problems → confirmed by inspecting completion text

**What this tells us for v15:**
- If H1 → v15 = dgxchen baseline + 1 single change (likely lm_head or multi-epoch); abandon augmenters entirely
- If H2 → revert cipher swap, keep dgxchen unchanged; v15 explores other single deltas
- If H3 → understand the dual-template fix Tong uses; possibly multi-adapter inference as v16+

**Reference scores:**
- v12 (Kh0a recipe, LB 0.72): 73.3% grader-correct (30-problem stratified)
- dgxchen (LB 0.84): see `dgxchen_eval_summary.json` from prior eval — should reproduce here
- v14 (LB 0.68): this run

In [ ]:
# Cell 1: Setup (sitedir + ptxas) — eval/inference needs ryanholbrook's patched transformers,
# unlike training notebooks which do NOT need site.addsitedir (see PLAN.md Rule #7).
import os, random, shutil, site, stat, sys, time, glob, json, math, re
import numpy as np
import torch

T0 = time.perf_counter()
def step(label):
    print(f"[{time.perf_counter() - T0:7.1f}s] {label}", flush=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Fail-fast GPU check (we keep getting P100 sometimes from CLI push — see Rule #8)
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA device")
gpu_name = torch.cuda.get_device_name(0)
gpu_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} ({gpu_vram_gb:.1f} GB)")
if gpu_vram_gb < 80:
    raise RuntimeError(f"Wrong GPU: {gpu_name} ({gpu_vram_gb:.1f} GB). Need RTX Pro 6000 (96 GB). Manual UI selection required.")

for utility_root in ("/kaggle/input/notebooks", "/kaggle/usr/lib/notebooks", "/kaggle/lib"):
    if not os.path.isdir(utility_root):
        continue
    for root, dirs, _ in os.walk(utility_root):
        if "models" in root.split(os.sep):
            dirs[:] = []
            continue
        if os.path.basename(root) == "python_packages":
            site.addsitedir(root)

PTXAS_DST = "/tmp/ptxas-blackwell"
for src in (
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell",
    "/kaggle/input/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell",
):
    if os.path.exists(src):
        if not os.path.exists(PTXAS_DST):
            shutil.copy(src, PTXAS_DST)
            os.chmod(PTXAS_DST, stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
        os.environ["TRITON_PTXAS_PATH"] = PTXAS_DST
        os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = PTXAS_DST
        break
step("Setup done")

In [ ]:
# Cell 2: Tokenizer + adapter paths (model load deferred to per-adapter cells)
#
# Lesson 2026-05-27: PEFT recent versions reject loading TWO adapters via
# multi-adapter API when both declare `target_parameters` (which Unsloth
# auto-adds for MoE: 'mlp.experts.gate_up_proj', 'mlp.experts.down_proj').
# Error: "Adding a LoRA config with target_parameters but there are already
# other LoRA adapters on this model that use target_parameters. At the moment,
# only one LoRA adapter per model with target_parameters is allowed."
#
# Workaround: load each adapter onto a FRESH base model. Costs ~8 min extra
# (one extra model load) but is the cleanest fix.

import kagglehub
import mamba_ssm  # noqa: F401
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
step(f"Tokenizer loaded; IM_END_ID={IM_END_ID}")

def find_adapter(*candidates):
    for p in candidates:
        if os.path.isdir(p) and os.path.exists(os.path.join(p, "adapter_config.json")):
            return p
    return None

DGXCHEN_PATH = find_adapter(
    "/kaggle/input/datasets/dgxchen/trained-adapter",
    "/kaggle/input/trained-adapter",
)
V14_PATH = find_adapter(
    "/kaggle/input/datasets/gideonnazarian/v14-c-adapter",
    "/kaggle/input/v14-c-adapter",
)

if DGXCHEN_PATH is None:
    raise FileNotFoundError("dgxchen adapter not found. /kaggle/input/ contents: " + str(os.listdir('/kaggle/input')))
if V14_PATH is None:
    hits = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
    raise FileNotFoundError(f"v14 adapter not found. adapter_config.json found at: {hits}")

step(f"dgxchen adapter at: {DGXCHEN_PATH}")
step(f"v14 adapter at:     {V14_PATH}")


def load_base_then_adapter(adapter_path: str):
    """Fresh base model + adapter. Returns the PeftModel ready for eval.
    Caller is responsible for `del`-ing the result + GC + cache clear before
    loading another adapter."""
    t = time.perf_counter()
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH, device_map="auto", trust_remote_code=True, dtype=torch.bfloat16,
    )
    step(f"Base model loaded in {time.perf_counter() - t:.0f}s")
    peft_m = PeftModel.from_pretrained(base, adapter_path)
    peft_m.eval()
    step(f"Adapter attached; VRAM={torch.cuda.memory_allocated() / 1e9:.1f} GB")
    return peft_m

In [ ]:
# Cell 3: Eval helpers + stratified holdout (the model param is now passed in)
import pandas as pd
from collections import defaultdict

PROMPT_SUFFIX = (
    "\nPlease put your final answer inside `\\boxed{}`. "
    "For example: `\\boxed{your answer}`"
)
SAMPLES_PER_CATEGORY = 5
MAX_NEW_TOKENS = 4096

TRAIN_CSV = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
df = pd.read_csv(TRAIN_CSV)

def classify(prompt: str) -> str:
    p = prompt.lower()
    if "bit manipulation" in p: return "bit_manipulation"
    if "secret encryption rules" in p: return "cipher"
    if "unit conversion" in p: return "unit_conversion"
    if "numeral system" in p: return "numeral_system"
    if "gravitational constant" in p: return "gravity"
    if "transformation rules" in p: return "transformation_rules"
    return "unknown"

df["category"] = df["prompt"].apply(classify)

holdout_ids = []
for cat in sorted(df["category"].unique()):
    if cat == "unknown": continue
    cat_df = df[df["category"] == cat]
    n = min(SAMPLES_PER_CATEGORY, len(cat_df))
    sampled = cat_df.sample(n=n, random_state=42)
    holdout_ids.extend(sampled["id"].tolist())
holdout = df[df["id"].isin(holdout_ids)].copy()
step(f"Stratified holdout: {len(holdout)} problems")
for cat in sorted(holdout["category"].unique()):
    print(f"  {cat}: {len(holdout[holdout['category']==cat])}")

BOXED_RE = re.compile(r"\\boxed\{([^}]*)\}")
THINK_CLOSE = "</think>"
IM_END = "<|im_end|>"
FLOAT_CATS = {"gravity", "unit_conversion"}

def extract_after_think(c):
    if THINK_CLOSE not in c: return ""
    post = c.split(THINK_CLOSE, 1)[1]
    if IM_END in post: post = post.split(IM_END, 1)[0]
    return post.strip()

def extract_boxed_anywhere(c):
    matches = list(BOXED_RE.finditer(c))
    return matches[-1].group(1).strip() if matches else ""

def is_grader_correct(boxed, gold, category):
    if not boxed or not isinstance(boxed, str): return False
    b = boxed.strip(); g = str(gold).strip()
    if category in FLOAT_CATS:
        try:
            return math.isclose(float(b), float(g), rel_tol=1e-2, abs_tol=1e-5)
        except (ValueError, TypeError):
            return b.lower() == g.lower()
    return b.lower() == g.lower()

def run_eval(model_obj, adapter_label: str):
    """Run the same holdout on the given PeftModel."""
    print(f"\n=== Eval: {adapter_label} ===")
    results = []
    for _, row in holdout.iterrows():
        pid = row["id"]; gold = str(row["answer"]); cat = row["category"]
        prompt_text = row["prompt"] + PROMPT_SUFFIX
        inputs = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt_text}],
            return_tensors="pt", add_generation_prompt=True, return_dict=True,
        ).to(model_obj.device)
        prompt_len = inputs["input_ids"].shape[1]
        t_p = time.perf_counter()
        with torch.no_grad():
            out = model_obj.generate(
                **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                eos_token_id=[IM_END_ID, tokenizer.eos_token_id],
                pad_token_id=tokenizer.eos_token_id,
            )
        elapsed = time.perf_counter() - t_p
        n_new = out.shape[1] - prompt_len
        completion = tokenizer.decode(out[0, prompt_len:], skip_special_tokens=False)
        after = extract_after_think(completion)
        boxed = extract_boxed_anywhere(completion)
        gc_correct = is_grader_correct(boxed, gold, cat)
        results.append({
            "adapter": adapter_label, "id": pid, "category": cat,
            "completion_tokens": n_new, "seconds": elapsed,
            "after_think": after, "boxed_anywhere": boxed, "gold": gold,
            "completion": completion,
            "has_think_close": THINK_CLOSE in completion,
            "correct_boxed": boxed.strip() == gold.strip(),
            "grader_correct": gc_correct,
            "correct_substring": bool(gold and gold in completion),
        })
        mark = "✓" if gc_correct else ("~" if (gold and gold in completion) else "✗")
        print(
            f"  [{len(results):2d}/{len(holdout)}] {cat:22s} {pid} | {n_new:4d} tok / {elapsed:5.1f}s | "
            f"boxed={boxed!r:25s} gold={gold!r:15s} {mark}",
            flush=True,
        )
    return results

In [ ]:
# Cell 4: Load dgxchen + eval, then unload completely (so v14 can load on fresh base)
import gc
peft_model = load_base_then_adapter(DGXCHEN_PATH)
dgxchen_results = run_eval(peft_model, "dgxchen")
step(f"dgxchen eval done")

# Tear down so Cell 5 can load v14 onto a clean base model
del peft_model
gc.collect()
torch.cuda.empty_cache()
step(f"Cleaned up dgxchen state; VRAM={torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# Cell 5: Load v14 onto fresh base + eval
peft_model = load_base_then_adapter(V14_PATH)
v14_results = run_eval(peft_model, "v14")
step(f"v14 eval done")

# Tear down before Cell 6 comparison
del peft_model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Cell 6: Side-by-side comparison + save artifacts
def per_category(results):
    pc = defaultdict(lambda: {"n": 0, "grader_correct": 0, "has_think_close": 0, "avg_tokens": 0.0})
    for r in results:
        c = r["category"]
        pc[c]["n"] += 1
        pc[c]["grader_correct"] += int(r["grader_correct"])
        pc[c]["has_think_close"] += int(r["has_think_close"])
        pc[c]["avg_tokens"] += r["completion_tokens"]
    for c in pc:
        pc[c]["avg_tokens"] = pc[c]["avg_tokens"] / max(pc[c]["n"], 1)
    return dict(pc)

dgx_pc = per_category(dgxchen_results)
v14_pc = per_category(v14_results)

print("\n=== SIDE-BY-SIDE COMPARISON ===")
print(f"{'category':22s} | dgxchen grader | v14 grader     | Δ      | dgx think | v14 think | dgx tok | v14 tok")
print("-" * 130)
all_cats = sorted(set(dgx_pc.keys()) | set(v14_pc.keys()))
for c in all_cats:
    d = dgx_pc.get(c, {"n":0,"grader_correct":0,"has_think_close":0,"avg_tokens":0})
    v = v14_pc.get(c, {"n":0,"grader_correct":0,"has_think_close":0,"avg_tokens":0})
    delta = v["grader_correct"] - d["grader_correct"]
    delta_s = f"{delta:+d}"
    print(
        f"{c:22s} | {d['grader_correct']}/{d['n']}            | "
        f"{v['grader_correct']}/{v['n']}            | {delta_s:6s} | "
        f"{d['has_think_close']}/{d['n']}       | {v['has_think_close']}/{v['n']}       | "
        f"{d['avg_tokens']:7.0f} | {v['avg_tokens']:7.0f}"
    )

n = len(dgxchen_results)
dgx_total = sum(d["grader_correct"] for d in dgx_pc.values())
v14_total = sum(v["grader_correct"] for v in v14_pc.values())
print(f"\nAggregate: dgxchen={dgx_total}/{n} = {dgx_total/n:.1%}, v14={v14_total}/{n} = {v14_total/n:.1%}")
print(f"LB reference: dgxchen=0.84, v14=0.68 (Δ=-0.16)")
print(f"Eval Δ:       {(v14_total-dgx_total)/n:+.1%}")

# Save
all_results = dgxchen_results + v14_results
pd.DataFrame(all_results).drop(columns=["completion"]).to_csv("/kaggle/working/v14_vs_dgxchen_results.csv", index=False)
with open("/kaggle/working/v14_vs_dgxchen_summary.json", "w") as f:
    json.dump({
        "n_total": n,
        "dgxchen_pc": dgx_pc,
        "v14_pc": v14_pc,
        "dgxchen_aggregate": dgx_total/n,
        "v14_aggregate": v14_total/n,
        "delta": (v14_total-dgx_total)/n,
    }, f, indent=2)
step(f"Saved v14_vs_dgxchen_results.csv + v14_vs_dgxchen_summary.json")

In [ ]:
# Cell 7: Side-by-side failure-mode inspection
# For each problem where v14 failed but dgxchen succeeded, print BOTH completions' tails.
# This is where we'll see whether v14 produced augmenter-format output instead of reasoning.
by_id = {}
for r in dgxchen_results:
    by_id[r["id"]] = {"dgx": r}
for r in v14_results:
    by_id[r["id"]]["v14"] = r

regressed = [pid for pid, both in by_id.items() if both["dgx"]["grader_correct"] and not both["v14"]["grader_correct"]]
improved = [pid for pid, both in by_id.items() if both["v14"]["grader_correct"] and not both["dgx"]["grader_correct"]]

print(f"Problems where v14 REGRESSED (dgx right, v14 wrong): {len(regressed)}")
print(f"Problems where v14 IMPROVED  (dgx wrong, v14 right): {len(improved)}\n")

for pid in regressed[:10]:  # first 10 for log size
    d = by_id[pid]["dgx"]
    v = by_id[pid]["v14"]
    print(f"=== {d['category']:22s} {pid} gold={d['gold']!r} ===")
    print(f"--- dgxchen (✓): boxed={d['boxed_anywhere']!r} tokens={d['completion_tokens']} ---")
    print(d['completion'][-500:])
    print(f"--- v14 (✗): boxed={v['boxed_anywhere']!r} tokens={v['completion_tokens']} ---")
    print(v['completion'][-500:])
    print()
step(f"Total wall: {(time.perf_counter() - T0) / 60:.1f} min")